In [1]:
!pip install LiteEFG numpy pandas tqdm

# Rock Paper Scissors — Imperfect Information EFG

## Structure
```
Level 1 — Player 1 chooses: Rock (R), Paper (P), or Scissors (S)
Level 2 — Player 2 chooses: Rock (R), Paper (P), or Scissors (S)
           Player 2 CANNOT see Player 1's choice — single infoset.
```

## Payoff Table (P1 / P2)
| P1 \ P2 | Rock | Paper | Scissors |
|---------|------|-------|----------|
| **Rock**     | 0/0  | -1/+1 | +1/-1    |
| **Paper**    | +1/-1| 0/0   | -1/+1    |
| **Scissors** | -1/+1| +1/-1 | 0/0      |

**Nash Equilibrium:** Both players should converge to uniform [1/3, 1/3, 1/3].

In [2]:
import LiteEFG as efg
from LiteEFG.baselines.CFR import graph as CFR
import pandas as pd
from tqdm import trange

# Write game file
game_content = """# Rock Paper Scissors IIEFG
#
# Opt {
#     game_tree: true,
#     num_players: 2,
#     depth: 2,
# }
#
node / player 1 actions R P S
node /P1:R player 2 actions R P S
node /P1:P player 2 actions R P S
node /P1:S player 2 actions R P S
node /P1:R/P2:R leaf payoffs 1=0  2=0
node /P1:R/P2:P leaf payoffs 1=-1 2=1
node /P1:R/P2:S leaf payoffs 1=1  2=-1
node /P1:P/P2:R leaf payoffs 1=1  2=-1
node /P1:P/P2:P leaf payoffs 1=0  2=0
node /P1:P/P2:S leaf payoffs 1=-1 2=1
node /P1:S/P2:R leaf payoffs 1=-1 2=1
node /P1:S/P2:P leaf payoffs 1=1  2=-1
node /P1:S/P2:S leaf payoffs 1=0  2=0
infoset pl1_0__ nodes /P1:R /P1:P /P1:S
infoset pl2_0__ nodes /P1:R /P1:P /P1:S
"""

game_path = "rps.game"
with open(game_path, "w") as f:
    f.write(game_content)
print("Game file written.")

# Load
env = efg.FileEnv(game_path, traverse_type="Enumerate")
print("FileEnv OK")

# CFR
alg = CFR()
print("CFR OK")

# Register
env.set_graph(alg)
print("set_graph OK — ready to train.")

Game file written.
FileEnv OK
===============Graph is ready for CFR===============


CFR OK
set_graph OK — ready to train.


In [3]:
# Run CFR
NUM_ITER   = 5000
PRINT_FREQ = 500
history    = []
best_exp   = 1e9

for i in trange(NUM_ITER, desc="CFR (RPS)"):
    alg.update_graph(env)
    env.update_strategy(alg.current_strategy(), update_best=False)

    if i % PRINT_FREQ == 0 or i == NUM_ITER - 1:
        explo = sum(env.exploitability(alg.current_strategy(), "avg-iterate"))
        best_exp = min(best_exp, explo)
        history.append({"iteration": i, "exploitability": round(explo, 8), "best": round(best_exp, 8)})

print("\nDone.")

CFR (RPS): 100%|████████████████████████| 5000/5000 [00:00<00:00, 259333.47it/s]


Done.


In [4]:
# Convergence table
print("--- Exploitability convergence ---")
print(pd.DataFrame(history).to_string(index=False))

--- Exploitability convergence ---
 iteration  exploitability  best
         0             0.0   0.0
       500             0.0   0.0
      1000             0.0   0.0
      1500             0.0   0.0
      2000             0.0   0.0
      2500             0.0   0.0
      3000             0.0   0.0
      3500             0.0   0.0
      4000             0.0   0.0
      4500             0.0   0.0
      4999             0.0   0.0


In [5]:
# Nash Equilibrium strategies
player_labels = {
    1: {"name": "Player 1", "actions": ["Rock", "Paper", "Scissors"]},
    2: {"name": "Player 2", "actions": ["Rock", "Paper", "Scissors"]},
}

print("=" * 55)
for player_id in range(1, 3):
    info = player_labels[player_id]
    print(f"\n{info['name']} — Nash Equilibrium (avg-iterate):")
    pairs = env.get_strategy(player_id, alg.current_strategy(), "avg-iterate")
    for infoset_name, probs in pairs:
        print(f"  Infoset: {infoset_name}")
        for action, p in zip(info['actions'], probs):
            bar = '█' * int(p * 30)
            print(f"    {action:10s}: {p:.4f}  ({p*100:.1f}%)  {bar}")
print("=" * 55)
print("\nExpected Nash: each action at ~33.3% for both players.")


Player 1 — Nash Equilibrium (avg-iterate):
  Infoset: pl1_1__singleton
    Rock      : 0.3333  (33.3%)  ██████████
    Paper     : 0.3333  (33.3%)  ██████████
    Scissors  : 0.3333  (33.3%)  ██████████

Player 2 — Nash Equilibrium (avg-iterate):
  Infoset: pl2_0__
    Rock      : 0.3333  (33.3%)  ██████████
    Paper     : 0.3333  (33.3%)  ██████████
    Scissors  : 0.3333  (33.3%)  ██████████

Expected Nash: each action at ~33.3% for both players.
